In [ ]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/23 11:36:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-silver"
SOURCE_SUSTEM = "postgres"
TABLE_NAME = "users_silver"


full_file_pathu = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{TABLE_NAME}.parquet"

In [4]:
users_df = spark.read.parquet(full_file_pathu).cache()
users_df.createOrReplaceTempView("users_table")

26/08/23 11:37:15 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [5]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-silver"
SOURCE_SUSTEM = "postgres"
TABLE_NAME = "subscriptions_silver"


full_file_paths = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{TABLE_NAME}.parquet"

In [6]:
subscriptions_df = spark.read.parquet(full_file_paths).cache()
subscriptions_df.createOrReplaceTempView("subscriptions_table")

In [10]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-silver"
SOURCE_SUSTEM = "postgres"
TABLE_NAME = "payments_silver"


full_file_pathp = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{TABLE_NAME}.parquet"

In [11]:
payments_df = spark.read.parquet(full_file_pathp).cache()
payments_df.createOrReplaceTempView("payments_table")

In [13]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-silver"
SOURCE_SUSTEM = "mongo"
TABLE_NAME = "videos_silver"


full_file_pathv = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{TABLE_NAME}.parquet"

In [14]:
videos_df = spark.read.parquet(full_file_pathv).cache()
videos_df.createOrReplaceTempView("videos_table")

26/08/23 12:26:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [17]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-silver"
SOURCE_SUSTEM = "kafka"
TABLE_NAME = "events_silver"


full_file_pathe = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{TABLE_NAME}.parquet"

In [18]:
events_df = spark.read.parquet(full_file_pathe).cache()
events_df.createOrReplaceTempView("events_table")

In [7]:
users_df.show(1,truncate=False,vertical=True)

-RECORD 0--------------------------------------------------------
 user_id                  | a760dcac-c679-49ac-ad49-113146639ec0 
 email                    | ychristensen@example.org             
 country_name             | Unknown                              
 account_status           | deleted                              
 created_at_timestamp     | 2025-09-26 08:07:20.248692           
 last_login_timestamp     | NULL                                 
 last_login_validated     | NULL                                 
 is_login_before_creation | 0                                    
 review_status            | NULL                                 
 has_email                | 1                                    
 has_logged_in            | 0                                    
 created_date             | 2025-09-26                           
 last_login_date          | NULL                                 
only showing top 1 row



In [8]:
subscriptions_df.show(1,truncate=False,vertical=True)

-RECORD 0-------------------------------------------------------------
 subscription_id               | d329c38d-f567-4b88-8ee6-aa0551130ab2 
 user_id                       | a760dcac-c679-49ac-ad49-113146639ec0 
 plan_type                     | free                                 
 start_date_timestamp          | 2025-08-29 21:16:17.6343             
 end_date_timestamp            | NULL                                 
 is_active                     | true                                 
 auto_renew                    | false                                
 end_date_validated            | NULL                                 
 is_end_date_before_start_date | 0                                    
 review_status                 | NULL                                 
 start_date                    | 2025-08-29                           
 end_date                      | NULL                                 
only showing top 1 row



In [15]:
payments_df.show(1,truncate=False,vertical=True)

[Stage 8:>                                                          (0 + 1) / 1]

-RECORD 0------------------------------------------------------
 payment_id             | 3296a3cd-2e51-4935-bed4-b4df646c49be 
 subscription_id        | 0b9484cf-8a16-43be-846a-d9266adc91c6 
 amount                 | 14.990000000000000000                
 currency               | USD                                  
 payment_date_timestamp | 2026-05-16 10:26:58.715617           
 payment_date           | 2026-05-16                           
 payment_status         | success                              
only showing top 1 row



In [16]:
videos_df.show(1,truncate=False,vertical=True)

[Stage 10:=============================>                            (1 + 1) / 2]

-RECORD 0----------------------------------------------------------------------------------------------------------------------
 video_id                      | 7a3e0a16-d4ea-4c06-a776-6a6a92579b5b                                                          
 user_id                       | 12e4f74a-a9a7-467d-93d3-17298cfebac0                                                          
 match_id                      | match_2070                                                                                    
 content_type                  | live                                                                                          
 status                        | archived                                                                                      
 title                         | Al_Khaleej vs Al_Bukiryah                                                                     
 description                   | Both decade happen opportunity bag. Teach always relate effect cut. Com

In [19]:
events_df.show(1,truncate=False,vertical=True)

-RECORD 0--------------------------------------------------------
 event_id                 | 8c6973d5-0dae-42ad-906a-a6c57685d7d8 
 user_id                  | 3dc19c4b-098c-4791-b993-93a22c544d97 
 video_id                 | 33ea938f-667b-439a-8e70-99cf6d78007f 
 interaction_type         | pause                                
 device_type              | mobile                               
 watch_time_sec           | NULL                                 
 event_timestamp          | 2026-07-13 19:23:14.473538           
 event_date               | 2026-07-13                           
 event_hour               | 19                                   
 dq_missing_watch_time    | false                                
 dq_unexpected_watch_time | false                                
 dq_invalid_watch_time    | false                                
 dq_invalid_interaction   | false                                
 dq_invalid_device        | false                                
 dq_invali

In [20]:
users_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- created_at_timestamp: timestamp (nullable = true)
 |-- last_login_timestamp: timestamp (nullable = true)
 |-- last_login_validated: timestamp (nullable = true)
 |-- is_login_before_creation: integer (nullable = true)
 |-- review_status: string (nullable = true)
 |-- has_email: integer (nullable = true)
 |-- has_logged_in: integer (nullable = true)
 |-- created_date: date (nullable = true)
 |-- last_login_date: date (nullable = true)



In [21]:
subscriptions_df.printSchema()

root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- start_date_timestamp: timestamp (nullable = true)
 |-- end_date_timestamp: timestamp (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- auto_renew: boolean (nullable = true)
 |-- end_date_validated: timestamp (nullable = true)
 |-- is_end_date_before_start_date: integer (nullable = true)
 |-- review_status: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)



In [22]:
payments_df.printSchema()

root
 |-- payment_id: string (nullable = true)
 |-- subscription_id: string (nullable = true)
 |-- amount: decimal(38,18) (nullable = true)
 |-- currency: string (nullable = true)
 |-- payment_date_timestamp: timestamp (nullable = true)
 |-- payment_date: date (nullable = true)
 |-- payment_status: string (nullable = true)



In [23]:
videos_df.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- match_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- created_date: date (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- last_updated_date: date (nullable = true)
 |-- competition_id: string (nullable = true)
 |-- competition_name: string (nullable = true)
 |-- competition_teams: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- technical_max_resolution: string (nullable = true)
 |-- technical_codec: string (nullable = true)
 |-- stats_views: integer (nullable = true)
 |-- stats_likes: integer (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (n

In [ ]:
events_df.printSchema()